# Check vs pythtb

**The primary gate on `modules/curvature.py`.** On the real Y2Ir2O7 base model, at a handful of k-points, checks that the batched k-k external curvature equals `TBModel.berry_curvature(include_external=True) - (include_external=False)` (pythtb's per-k implementation, itself validated against postw90 `J0+J1`), using the same eigenvector/denominator conventions as `calculations/Y2Ir2O7/phonon/run_axion.py`. Also exercises the mode model's `_r.dat` fallback and validates the mixed k-beta assembly with nonzero `A_beta`.

This check has **no `results/` artifacts** -- it is pure assertions, not a persisted measurement, so there is nothing to reload. Set `RUN_COMPUTATION = True` and run every cell to exercise it; it parses two full Y2Ir2O7 `tb.dat` files and takes a few minutes.

In [1]:
import sys
from pathlib import Path

import numpy as np
from pythtb import W90

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from modules import curvature as cv
from modules.axion import beta_terms, parametric_connection, prepare_lambda
from validation._paths import (
    YIO_BASE,
    YIO_D_BETA,
    YIO_MODE1,
    YIO_N_OCCUPIED,
)

## Config

Same base/mode pair and band count as the production sweep in `calculations/Y2Ir2O7/phonon/run_axion.py`: trial_02 (Y:p, Ir:d, O:s;p), 176 WF, n_occ = 156. The legacy 196-orbital pair (base `.../173260bc889`, mode `.../output/173195bc889`, N_OCC = 104) is a different Wannier set and does not test the models in use.

In [2]:
RUN_COMPUTATION = False  # True: parse both tb.dat files and run every check below (several minutes).
BASE_DIR = YIO_BASE
MODE_DIR = YIO_MODE1
N_OCC = YIO_N_OCCUPIED
D_BETA = YIO_D_BETA

## Test k-points

A few high-symmetry points plus three random ones.

In [3]:
rng = np.random.default_rng(7)
k_test = np.vstack(
    [
        [0.0, 0.0, 0.0],
        [0.5, 0.0, 0.0],
        [0.25, 0.25, 0.25],
        rng.uniform(0, 1, size=(3, 3)),
    ]
)

## Load the base W90 model

In [4]:
if RUN_COMPUTATION:
    print("Loading base W90 model (parses tb.dat, may take a few minutes)...")
    w90_base = W90(str(BASE_DIR), "Y2Ir2O7")
    model_base = w90_base.model(
        zero_energy=0, min_hopping_norm=1e-5, max_distance=125,
        ignorable_imaginary_part=None,
    )
    assert model_base._pos_r is not None, "base model lost its position matrix"
    print(f"  norb = {model_base.norb}, n_R(pos) = {len(model_base._pos_r)}")

    occ = np.arange(N_OCC)
    cond = np.setdiff1d(np.arange(model_base.norb), occ)

Loading base W90 model (parses tb.dat, may take a few minutes)...


  norb = 176, n_R(pos) = 617


## pythtb reference curvature (internal vs internal+external)

`include_external=None` (the default) auto-enables for W90 models carrying a position matrix, so the internal-only reference must say `False` explicitly.

In [5]:
if RUN_COMPUTATION:
    print("Computing pythtb reference (internal and internal+external)...")
    ref_tot = model_base.berry_curvature(
        k_test, occ_idxs=occ, non_abelian=True, include_external=True
    )
    ref_int = model_base.berry_curvature(
        k_test, occ_idxs=occ, non_abelian=True, include_external=False
    )
    ref_ext = ref_tot - ref_int  # (3, 3, Nk, n_occ, n_occ)

Computing pythtb reference (internal and internal+external)...


## Our batched external curvature (k-k block), pipeline conventions

In [6]:
if RUN_COMPUTATION:
    print("Computing batched external terms...")
    H = model_base.hamiltonian(k_test)  # (Nk, N, N), convention I
    v = model_base.velocity(k_test)  # (3, Nk, N, N), reduced (cartesian=False)

    E, U = np.linalg.eigh(H)
    evecs_rows = U.swapaxes(-1, -2)  # state index first, as in the TF pipeline
    Vc, Vt = evecs_rows.conj(), evecs_rows.swapaxes(-1, -2)
    v_rot = Vc[None] @ v @ Vt[None]  # (3, Nk, N, N)

    # d_{cm} = v_{cm} / (E_m - E_c): conduction-occupied block over energy denom.
    inv_delta_co = 1.0 / (E[:, occ][:, None, :] - E[:, cond][:, :, None])  # (Nk, nc, no)
    d_co = v_rot[:, :, cond[:, None], occ[None, :]] * inv_delta_co[None]

    # position_terms does what pythtb does: X[R=0] -= tau_cart, keeping the residual
    # <0s|r|0s> - tau_s (~5e-7 Angstrom against the _r.dat print precision).
    pos = cv.position_terms(model_base, dtype=np.complex128)
    f = cv.fields(model_base, pos, k_test)
    A_red = f.A.swapaxes(0, 1)  # (Nk, 3, N, N): k first, as the hand assembly below indexes it
    c = cv.connection(f, N_OCC)
    ext_mine = cv.omega_cross(c) + cv.omega_external(c)  # pythtb's include_external=True minus False

Computing batched external terms...


## Compare the k-k block

In [7]:
if RUN_COMPUTATION:
    scale = np.max(np.abs(ref_ext))
    err = np.max(np.abs(ext_mine - ref_ext))
    print(f"  max |batched - pythtb| = {err:.3e}   (scale: max|ref| = {scale:.3e})")
    assert err < 1e-8 * max(scale, 1.0), "k-k external terms do not match pythtb"
    print("  k-k external terms MATCH pythtb reference.")

  max |batched - pythtb| = 5.722e-17   (scale: max|ref| = 6.109e-01)
  k-k external terms MATCH pythtb reference.


## Mode model via the `_r.dat` fallback

Built directly on the base embedding -- it cannot be applied afterwards, since `TBModel.lattice` returns a copy and assigning through it is a silent no-op (pythtb/tbmodel.py:233).

In [8]:
if RUN_COMPUTATION:
    print("Loading mode W90 model (parses r.dat fallback, may take a few minutes)...")
    w90_mode = W90(str(MODE_DIR), "Y2Ir2O7")
    nominal_orb_vecs = model_base.orb_vecs.copy()
    model_mode = w90_mode.model(
        zero_energy=0, min_hopping_norm=1e-5, max_distance=125,
        ignorable_imaginary_part=None, orb_vecs=nominal_orb_vecs,
    )
    assert np.allclose(model_mode.orb_vecs, nominal_orb_vecs, atol=1e-12), (
        "mode model did not take the shared embedding"
    )
    assert model_mode._pos_r is not None, (
        "mode model has no position matrix - r.dat fallback failed"
    )
    print(f"  mode n_R(pos) = {len(model_mode._pos_r)}")

    pos_mode = cv.position_terms(model_mode, dtype=np.complex128)
    f_mode = cv.fields(model_mode, pos_mode, k_test)

Loading mode W90 model (parses r.dat fallback, may take a few minutes)...


  mode n_R(pos) = 617


## Lambda(0) perturbation and the mixed k-beta connection

A small anti-Hermitian `Lambda(0)` gives a nonzero Hermitian `A_beta = i Lambda`; an off-diagonal entry exercises `partial_i A_beta` too.

In [9]:
if RUN_COMPUTATION:
    lambda_0 = 1j * np.diag(np.linspace(0.01, 0.02, model_base.norb))
    lambda_0[0, 1] = 0.03
    lambda_0[1, 0] = -0.03
    A_beta, dA_beta = parametric_connection(pos, prepare_lambda({(0, 0, 0): lambda_0}, np.complex128), k_test)
    beta = beta_terms(f, f_mode, D_BETA, A_beta, dA_beta)  # dH/dbeta, dA/dbeta, mixed curl (35)
    assert np.max(np.abs(beta.A)) > 0.0

## Full 4x4 mixed-plane curvature

`ext4[:3, :3]` must reproduce the k-k-only `ext_mine` unchanged by the mixed-plane path.

In [10]:
if RUN_COMPUTATION:
    # H_mode already carries the shared embedding (applied at build time above).
    H1 = model_mode.hamiltonian(k_test)
    v_beta = (H1 - H) / D_BETA
    vb_rot = Vc @ v_beta @ Vt
    d4_co = np.concatenate(
        [d_co, (vb_rot[:, cond[:, None], occ[None, :]] * inv_delta_co)[None]], axis=0
    )

    c4 = cv.connection(f, N_OCC, beta)
    ext4 = cv.omega_cross(c4) + cv.omega_external(c4)
    assert ext4.shape == (4, 4, k_test.shape[0], N_OCC, N_OCC)
    assert np.allclose(ext4[:3, :3], ext_mine), "k-k block changed by mixed-plane path"
    ext4.shape

## Hand-assembled cross-term check

Independently assembles the full mixed formula: all four cross terms, the ordinary curl, and `-i[A_i, A_beta]`. Does not reuse `curvature.omega_cross` / `omega_external`'s own bookkeeping.

In [11]:
if RUN_COMPUTATION:
    def adjoint(X):
        return X.conj().swapaxes(-1, -2)

    a_k = Vc[:, None] @ A_red @ Vt[:, None]
    a_beta = (Vc[:, None] @ beta.A[:, None] @ Vt[:, None])[:, 0]
    mixed_curl = Vc[:, None] @ beta.curl.swapaxes(0, 1) @ Vt[:, None]
    a_k_co = a_k[:, :, cond[:, None], occ[None, :]]
    a_k_oo = a_k[:, :, occ[:, None], occ[None, :]]
    a_beta_co = a_beta[:, cond[:, None], occ[None, :]]
    a_beta_oo = a_beta[:, occ[:, None], occ[None, :]]
    mixed_curl_oo = mixed_curl[:, :, occ[:, None], occ[None, :]]

    for i in range(3):
        x_i = -1j * a_k_co[:, i]
        x_beta = -1j * a_beta_co
        cross = 1j * (
            adjoint(d4_co[i]) @ x_beta
            + adjoint(x_i) @ d4_co[3]
            - adjoint(d4_co[3]) @ x_i
            - adjoint(x_beta) @ d4_co[i]
        )
        pure_external = mixed_curl_oo[:, i] - 1j * (
            a_k_oo[:, i] @ a_beta_oo - a_beta_oo @ a_k_oo[:, i]
        )
        assert np.allclose(ext4[i, 3], cross + pure_external)

## Hermiticity and magnitude summary

In [12]:
if RUN_COMPUTATION:
    herm = np.max(np.abs(ext4 - np.transpose(ext4, (0, 1, 2, 4, 3)).conj()))
    print(f"  full mixed-plane block computed with nonzero A_beta: shape {ext4.shape}")
    print(f"  hermiticity residual of curvature blocks: {herm:.3e}")
    print(f"  |ext k-k|_max = {np.max(np.abs(ext4[:3, :3])):.3e}, "
          f"|ext k-beta|_max = {np.max(np.abs(ext4[:3, 3])):.3e}")
    print("\nALL CHECKS PASSED")

  full mixed-plane block computed with nonzero A_beta: shape (4, 4, 6, 156, 156)
  hermiticity residual of curvature blocks: 5.165e-15
  |ext k-k|_max = 6.109e-01, |ext k-beta|_max = 1.476e+00

ALL CHECKS PASSED
